# NB136 — The Four Mass Channels

## Purpose

Collect and systematize the four physical mass channels that convert cascade CP ratios
into fermion mass ratios. Each channel uses a specific cascade level, a specific algebraic
exponent, and a specific correction factor (if any). Together, they produce ALL charged
fermion masses from $M_Z$ with zero free parameters.

## Status entering
- 277 identities, 0 free parameters, 135 notebooks
- NB135: #277 promoted to PASS — $x_{\mu/e} = p_2 = 3$ (lepton intra-gen)
- NB124: $m_\tau/m_\mu = C_0^{x_3} \times p_3/p_4$ (#269 PASS)
- NB127: $m_t/m_b = P_4/p_3 = 42$ (#271 PASS)
- NB118: $m_t/M_Z = p_2^2/\sqrt{\pi p_4}$ (#258 PASS)
- NB134: Quark window-0 exponent $x_{s/d} = 1.586646 \approx 2^{2/3}$ (not promoted)

## Open questions
1. What is the algebraic exponent for the quark intra-gen channel?
2. Why do $x_{s/d}$ and $x_{\tau/\mu}^{\text{eff}}$ nearly coincide at $\approx 1.587$?
3. Can all 9 charged fermion masses be tabulated from $M_Z$ alone?

## Sections
- **S0**: Setup and integration
- **S1**: Four-channel inventory with window-0 CP ratios
- **S2**: Quark intra-gen exponent — algebraic candidate battery
- **S3**: The exponent coincidence — $x_{s/d} \approx x_{\tau/\mu}^{\text{eff}}$
- **S4**: Complete 9-mass table from $M_Z$
- **S5**: Synthesis and verdict

## Identity targets: #278+

In [3]:

# -- S0: Setup and integration --
import sys, numpy as np, time
from pathlib import Path
from fractions import Fraction
from math import gcd

ROOT = Path.cwd().parent
if str(ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(ROOT / 'scripts'))

from solenoid_algebra import (SA, RHO, KAPPA, EPSILON, OMEGA,
                               X4, X3, X2, LAM7, X4_LEP, GAMMA,
                               CP_PAIRS, SM_TARGETS, PHYSICAL_CROSSINGS)
from solenoid_system import SolenoidSystem
from solenoid_jax import warmup as jax_warmup, detect_device

p1, p2, p3, p4 = SA.primes
P4 = SA.P           # 210
PHI_P4 = SA.PHI     # 48
LAM_P4 = SA.group_exponent  # 12

def target_value(key):
    """Extract float from SM_TARGETS (may be tuple or float)."""
    v = SM_TARGETS[key]
    return v[0] if isinstance(v, (tuple, list)) else float(v)

# SM targets
M_Z = 91.1876  # GeV
M_MU_E   = target_value('m_mu/m_e')
M_TAU_MU = target_value('m_tau/m_mu')
M_TAU_E  = target_value('m_tau/m_e')
M_S_D    = target_value('m_s/m_d')
M_T_B    = target_value('m_t/m_b')
M_T_C    = target_value('m_t/m_c')
M_B_S    = target_value('m_b/m_s')
M_C_U    = target_value('m_c/m_u')

# System
sys0 = SolenoidSystem()
all_branches = sys0.all_branches()

# JAX warmup
print(f'JAX device: {detect_device()}')
t0 = time.time()
jax_warmup()
print(f'JAX warmup: {time.time()-t0:.1f}s')

# Integrate
T_MAX = 5000
cis = SA.coprime_indices(T_MAX)
t_cross = cis.astype(float)
T_integ = float(T_MAX + 1)
a3_t, a5_t, a7_t = SA.sector_labels(cis)

print(f'\nIntegrating {len(all_branches)} branches to T={T_MAX}...')
t0 = time.time()
res = sys0.integrate_all_branches(all_branches, t_cross, T_integ, backend='jax')
dt = time.time() - t0
print(f'Done in {dt:.1f}s. {len(cis)} crossings.')

# Window-0 extraction (returns tuple: (ratios_dict, sector_rms))
w0_cp, w0_sector_rms = SolenoidSystem.window0_cp_ratios(
    res, cis, a3_t, a5_t, a7_t, P=P4, n_levels=4, cp_pairs=CP_PAIRS)

# Report
print(f'\nNB136: THE FOUR MASS CHANNELS')
print('=' * 65)
print(f'  P4 = {P4}, phi(P4) = {PHI_P4}, lambda(P4) = {LAM_P4}')
print(f'  rho = kappa = 1/sqrt({P4}) = {RHO:.6f}')
print(f'  Dissipation gamma_k = p_k^2: {[g for g in GAMMA]}')
print(f'  Exponents: x3={X3:.6f}, x4={X4:.6f}, x4_lep={X4_LEP:.6f}, x2={X2:.6f}')
print(f'\nWindow-0 CP ratios:')
for ch in ['QUARK', 'LEPTON']:
    vals = w0_cp[ch]
    print(f'  {ch:>7}: R0={vals[0]:.4f}  R1={vals[1]:.4f}  R2={vals[2]:.4f}  R3={vals[3]:.4f}')


JAX device: CPU (1 device(s))
JAX warmup: 0.7s

Integrating 210 branches to T=5000...
  JAX [CPU (1 device(s))]: 210 branches, 1143 eval pts, T=5001.0 — 26.22s
Done in 26.2s. 1143 crossings.

NB136: THE FOUR MASS CHANNELS
  P4 = 210, phi(P4) = 48, lambda(P4) = 12
  rho = kappa = 1/sqrt(210) = 0.069007
  Dissipation gamma_k = p_k^2: [4, 9, 25, 49]
  Exponents: x3=1.909859, x4=7.639437, x4_lep=7.798592, x2=1.273240

Window-0 CP ratios:
    QUARK: R0=189.1119  R1=58.8635  R2=39.8014  R3=6.6067
   LEPTON: R0=8.7738  R1=5.4299  R2=5.2273  R3=5.9120


## S1: Four-Channel Inventory

The four physical mass channels, each with its established formula:

| # | Channel | Level | Exponent | Correction | Formula |
|---|---------|-------|----------|------------|--------|
| 1 | m_μ/m_e (lepton intra-gen) | R₃ (index 3) | p₂ = 3 | none | C₀_l³ |
| 2 | m_τ/m_μ (lepton inter-gen) | R₂ (index 2) | x₃ = λ(P₄)/(2π) | p₃/p₄ | C₀_l^x₃ × 5/7 |
| 3 | m_s/m_d (quark intra-gen) | R₃ (index 3) | x ≈ 1.587 | none? | C₀_q^x |
| 4 | m_t/m_b (algebraic) | — | — | — | P₄/p₃ = 42 |

Note: "R₃ (index 3)" means the outermost cascade level, which is R₃ in the
4-level system (indices 0,1,2,3). The naming can be confusing because some
notebooks call it R₄ (counting from 1). We use array index 3 = outermost.

In [4]:
# -- S1: Four-channel inventory --

# Extract the key window-0 CP ratios
C0_l_R3 = w0_cp['LEPTON'][3]  # outermost level, lepton (for mu/e)
C0_l_R2 = w0_cp['LEPTON'][2]  # second-outermost, lepton (for tau/mu)
C0_q_R3 = w0_cp['QUARK'][3]   # outermost level, quark (for s/d)
C0_q_R2 = w0_cp['QUARK'][2]   # second-outermost, quark

print('S1: FOUR-CHANNEL INVENTORY')
print('=' * 70)
print()

# Channel 1: Lepton intra-gen (mu/e)
pred_mu_e = C0_l_R3 ** p2
dev_mu_e = (pred_mu_e / M_MU_E - 1) * 100
print(f'CHANNEL 1: LEPTON INTRA-GEN (m_mu/m_e)')
print(f'  Formula: C0_l_R3^p2 = {C0_l_R3:.6f}^{p2} = {pred_mu_e:.3f}')
print(f'  SM:      {M_MU_E:.3f}')
print(f'  Dev:     {dev_mu_e:+.4f}%  [#277 PASS]')
print()

# Channel 2: Lepton inter-gen (tau/mu)
pred_tau_mu = C0_l_R2 ** X3 * p3 / p4
dev_tau_mu = (pred_tau_mu / M_TAU_MU - 1) * 100
print(f'CHANNEL 2: LEPTON INTER-GEN (m_tau/m_mu)')
print(f'  Formula: C0_l_R2^x3 * p3/p4 = {C0_l_R2:.6f}^{X3:.6f} * {p3}/{p4}')
print(f'         = {C0_l_R2**X3:.4f} * {p3/p4:.6f} = {pred_tau_mu:.4f}')
print(f'  SM:      {M_TAU_MU:.3f}')
print(f'  Dev:     {dev_tau_mu:+.4f}%  [#269 PASS]')
print()

# Channel 3: Quark intra-gen (s/d)
x_eff_q = np.log(M_S_D) / np.log(C0_q_R3)
pred_sd_cuberoot4 = C0_q_R3 ** (2**(2/3))
dev_sd_cr4 = (pred_sd_cuberoot4 / M_S_D - 1) * 100
print(f'CHANNEL 3: QUARK INTRA-GEN (m_s/m_d)')
print(f'  C0_q_R3 = {C0_q_R3:.6f}')
print(f'  Effective exponent: x_eff = ln({M_S_D:.2f})/ln({C0_q_R3:.6f}) = {x_eff_q:.6f}')
print(f'  Candidate: 2^(2/3) = {2**(2/3):.6f}')
print(f'  C0^(2^(2/3)) = {pred_sd_cuberoot4:.4f}  (SM: {M_S_D:.2f}, dev: {dev_sd_cr4:+.4f}%)')
print(f'  Status: NOT YET ESTABLISHED — exponent identity unknown')
print()

# Channel 4: Algebraic cross-sector (t/b)
m_t_sol = M_Z * p2**2 / np.sqrt(np.pi * p4)
m_b_sol = m_t_sol / (P4 // p3)
print(f'CHANNEL 4: ALGEBRAIC CROSS-SECTOR')
print(f'  m_t/M_Z = p2^2/sqrt(pi*p4) = {p2**2}/sqrt({np.pi*p4:.4f}) = {m_t_sol/M_Z:.6f}')
print(f'  m_t = {m_t_sol:.2f} GeV  (#258 PASS)')
print(f'  m_t/m_b = P4/p3 = {P4}/{p3} = {P4//p3}  (#271 PASS)')
print(f'  m_b = {m_b_sol:.4f} GeV')
print()

# Combined lepton: m_tau/m_e
pred_tau_e = pred_tau_mu * pred_mu_e
dev_tau_e = (pred_tau_e / M_TAU_E - 1) * 100
print(f'COMBINED: m_tau/m_e = {pred_tau_mu:.4f} * {pred_mu_e:.3f} = {pred_tau_e:.1f}')
print(f'  SM: {M_TAU_E:.1f}, dev: {dev_tau_e:+.3f}%')

S1: FOUR-CHANNEL INVENTORY

CHANNEL 1: LEPTON INTRA-GEN (m_mu/m_e)
  Formula: C0_l_R3^p2 = 5.911955^3 = 206.630
  SM:      206.768
  Dev:     -0.0668%  [#277 PASS]

CHANNEL 2: LEPTON INTER-GEN (m_tau/m_mu)
  Formula: C0_l_R2^x3 * p3/p4 = 5.227295^1.909859 * 5/7
         = 23.5401 * 0.714286 = 16.8143
  SM:      16.817
  Dev:     -0.0158%  [#269 PASS]

CHANNEL 3: QUARK INTRA-GEN (m_s/m_d)
  C0_q_R3 = 6.606742
  Effective exponent: x_eff = ln(20.00)/ln(6.606742) = 1.586646
  Candidate: 2^(2/3) = 1.587401
  C0^(2^(2/3)) = 20.0285  (SM: 20.00, dev: +0.1426%)
  Status: NOT YET ESTABLISHED — exponent identity unknown

CHANNEL 4: ALGEBRAIC CROSS-SECTOR
  m_t/M_Z = p2^2/sqrt(pi*p4) = 9/sqrt(21.9911) = 1.919193
  m_t = 175.01 GeV  (#258 PASS)
  m_t/m_b = P4/p3 = 210/5 = 42  (#271 PASS)
  m_b = 4.1668 GeV

COMBINED: m_tau/m_e = 16.8143 * 206.630 = 3474.3
  SM: 3477.2, dev: -0.082%


## S2: The Quark Intra-Gen Exponent

NB134-135 established that the lepton intra-gen exponent is exactly $p_2 = 3$.
The quark intra-gen exponent $x_{s/d} = 1.586646$ is T-independent but has no
identified algebraic form. Candidates to test systematically:

- $2^{2/3} = \sqrt[3]{4} = 1.587401$ (+0.048%)
- $p_3/\pi = 5/\pi = 1.591549$ (+0.309%)
- $\ln(p_3) = 1.609438$ (+1.438%)
- $X_3/X_2 = (12/8) = 1.5$ (−5.46%)
- Algebraic functions of the primes

The key test: match the measured $x_{\text{eff}}$ to an expression from $\{2,3,5,7\}$.

In [5]:
# -- S2: Quark intra-gen exponent — algebraic candidate battery --
from sympy import Rational, sqrt as sym_sqrt, pi as sym_pi, log as sym_log
from sympy import cbrt, totient

x_measured = x_eff_q  # from S1

# Generate many algebraic candidates from {2,3,5,7}
candidates = {}

# Simple prime ratios and powers
candidates['p2 = 3'] = 3.0
candidates['p2/p1 = 3/2'] = 1.5
candidates['2^(2/3) = cbrt(4)'] = 2**(2/3)
candidates['p3/pi = 5/pi'] = 5 / np.pi
candidates['p4/pi^2 = 7/pi^2'] = 7 / np.pi**2
candidates['ln(p3) = ln(5)'] = np.log(5)
candidates['ln(p3)/ln(p2) = ln5/ln3'] = np.log(5)/np.log(3)
candidates['p1^(p2/p3) = 2^(3/5)'] = 2**(3/5)
candidates['p2^(p1/p3) = 3^(2/5)'] = 3**(2/5)
candidates['p3^(p1/p4) = 5^(2/7)'] = 5**(2/7)
candidates['p2^(1/p1) = sqrt(3)'] = np.sqrt(3)
candidates['phi(P4)^(1/p4) = 48^(1/7)'] = 48**(1/7)
candidates['(p2*p3)^(1/p2) = 15^(1/3)'] = 15**(1/3)
candidates['P4^(1/p4) = 210^(1/7)'] = 210**(1/7)
candidates['lam(P4)^(1/p3) = 12^(1/5)'] = 12**(1/5)
candidates['phi(P3)/(2*pi) = 8/(2*pi) = X2'] = X2
candidates['X3/X2 = 12/8 = 3/2'] = X3 / X2
candidates['(X4 + X4_LEP)/2/pi'] = (X4 + X4_LEP) / (2*np.pi)
candidates['phi(P4)/P3 = 48/30 = 8/5'] = 8/5
candidates['lam(P4)/phi(p4) = 12/6 = 2'] = 2.0
candidates['p1*phi(p4)/(2*pi) = 12/(2*pi)'] = 12/(2*np.pi)  # = X3
candidates['(p2+p3)/p3 = 8/5'] = (p2 + p3) / p3
candidates['p4/(p1*p1*pi) = 7/(4*pi)'] = 7 / (4*np.pi)
candidates['sqrt(p3/p1) = sqrt(5/2)'] = np.sqrt(5/2)
candidates['p4/(p2*pi) = 7/(3*pi)'] = 7 / (3*np.pi)

# Sort by closeness to measured value
scored = [(name, val, (val/x_measured - 1) * 100)
          for name, val in candidates.items()]
scored.sort(key=lambda x: abs(x[2]))

print('S2: QUARK INTRA-GEN EXPONENT — ALGEBRAIC CANDIDATES')
print('=' * 70)
print(f'  Measured x_eff = {x_measured:.6f}')
print()
print(f'  {"Expression":>35} {"Value":>10} {"Dev from x_eff":>14}')
print(f'  {"-"*62}')
for name, val, dev in scored[:15]:
    print(f'  {name:>35} {val:>10.6f} {dev:>+13.4f}%')

# The best candidate: 2^(2/3)
best_val = 2**(2/3)
print(f'\nBEST CANDIDATE: 2^(2/3) = cbrt(4) = {best_val:.6f}')
print(f'  Deviation from measured: {(best_val/x_measured - 1)*100:+.4f}%')
print(f'  Mass prediction: C0^(2^(2/3)) = {C0_q_R3**best_val:.4f}')
print(f'  SM target: {M_S_D:.2f}')
print(f'  Mass dev: {(C0_q_R3**best_val / M_S_D - 1)*100:+.4f}%')

# Compare to the lepton exponent quality
x_lep_measured = np.log(M_MU_E) / np.log(C0_l_R3)
print(f'\nCOMPARISON TO LEPTON CHANNEL:')
print(f'  Lepton x_eff = {x_lep_measured:.10f}, target p2 = 3, dev = {(x_lep_measured/3-1)*100:+.6f}%')
print(f'  Quark  x_eff = {x_measured:.10f}, target 2^(2/3) = {best_val:.10f}, dev = {(x_measured/best_val-1)*100:+.6f}%')
print(f'  Lepton precision: {abs(x_lep_measured/3-1)*1e6:.1f} ppm')
print(f'  Quark  precision: {abs(x_measured/best_val-1)*1e6:.1f} ppm')
print(f'  Quark is ~{abs(x_measured/best_val-1)/abs(x_lep_measured/3-1):.0f}x less precise than lepton')

S2: QUARK INTRA-GEN EXPONENT — ALGEBRAIC CANDIDATES
  Measured x_eff = 1.586646

                           Expression      Value Dev from x_eff
  --------------------------------------------------------------
                    2^(2/3) = cbrt(4)   1.587401       +0.0476%
                 p3^(p1/p4) = 5^(2/7)   1.583820       -0.1782%
                         p3/pi = 5/pi   1.591549       +0.3090%
              sqrt(p3/p1) = sqrt(5/2)   1.581139       -0.3471%
             phi(P4)/P3 = 48/30 = 8/5   1.600000       +0.8416%
                     (p2+p3)/p3 = 8/5   1.600000       +0.8416%
                       ln(p3) = ln(5)   1.609438       +1.4365%
                 p2^(p1/p3) = 3^(2/5)   1.551846       -2.1934%
            lam(P4)^(1/p3) = 12^(1/5)   1.643752       +3.5991%
                 p1^(p2/p3) = 2^(3/5)   1.515717       -4.4704%
                          p2/p1 = 3/2   1.500000       -5.4610%
                   X3/X2 = 12/8 = 3/2   1.500000       -5.4610%
              ln(p3)/l

## S3: The Exponent Coincidence

NB134 observed that the quark intra-gen exponent ($x_{s/d} = 1.5866$) and the
tau/mu effective exponent ($x_{\text{eff}} = \ln(16.817)/\ln(C_0) = 1.5883$) nearly
coincide. Is this a structural connection or a numerical accident?

The NB124 tau/mu formula decomposes as $C_0^{x_3} \times p_3/p_4$. What effective
exponent does this correspond to?

In [7]:

# -- S3: The exponent coincidence --

print('S3: THE EXPONENT COINCIDENCE')
print('=' * 70)
print()

# NB134 defined x_eff for EACH channel as: ln(target) / ln(C0_R3)
# i.e., project everything onto the outermost level.
# This is where the coincidence lives.

x_sd_R3 = np.log(M_S_D) / np.log(C0_q_R3)  # quark intra-gen on R3
x_mu_e_R3 = np.log(M_MU_E) / np.log(C0_l_R3)  # lepton intra-gen on R3 = p2 = 3
x_tau_mu_R3 = np.log(M_TAU_MU) / np.log(C0_l_R3)  # tau/mu projected onto R3

print(f'ALL EFFECTIVE EXPONENTS PROJECTED ONTO OUTERMOST C0:')
print(f'  Lepton R3 C0 = {C0_l_R3:.6f}')
print(f'  Quark  R3 C0 = {C0_q_R3:.6f}')
print()
print(f'  x_eff(mu/e)   = ln({M_MU_E:.2f})/ln({C0_l_R3:.6f}) = {x_mu_e_R3:.6f}  [IDENTIFIED: p2 = 3]')
print(f'  x_eff(tau/mu) = ln({M_TAU_MU:.3f})/ln({C0_l_R3:.6f}) = {x_tau_mu_R3:.6f}  [actual mechanism: R2 level]')
print(f'  x_eff(s/d)    = ln({M_S_D:.2f})/ln({C0_q_R3:.6f}) = {x_sd_R3:.6f}  [OPEN]')
print()

# The NB134 coincidence: x_sd ≈ x_tau_mu when both are on R3
diff_pct = (x_tau_mu_R3 / x_sd_R3 - 1) * 100
print(f'THE NB134 COINCIDENCE (both projected onto outermost level):')
print(f'  x_eff(s/d)    = {x_sd_R3:.6f}')
print(f'  x_eff(tau/mu) = {x_tau_mu_R3:.6f}')
print(f'  Difference:     {diff_pct:+.4f}%')
print()

# But the tau/mu mechanism is COMPLETELY DIFFERENT:
# NB124: m_tau/m_mu = C0_l_R2^x3 * p3/p4
# So x_tau_mu projected onto R3 is an ARTIFACT of projection.
# It equals: x3 * ln(C0_R2)/ln(C0_R3) + ln(p3/p4)/ln(C0_R3)
x_tau_mu_actual = X3  # the actual algebraic exponent (on R2)
x_tau_mu_correction = np.log(p3/p4) / np.log(C0_l_R2)  # on R2

print(f'DIAGNOSIS: WHY THE COINCIDENCE EXISTS')
print(f'  The tau/mu actual mechanism (NB124): C0_R2^x3 * p3/p4')
print(f'    Algebraic exponent x3 = {X3:.6f} (on R2, C0 = {C0_l_R2:.6f})')
print(f'    Correction: p3/p4 = {p3/p4:.6f}')
print()
print(f'  When PROJECTED onto R3:')
print(f'    x_eff(tau/mu) = x3 * ln(C0_R2)/ln(C0_R3) + ln(p3/p4)/ln(C0_R3)')
cross_factor = np.log(C0_l_R2)/np.log(C0_l_R3)
corr_factor = np.log(p3/p4)/np.log(C0_l_R3)
print(f'                  = {X3:.6f} * {cross_factor:.6f} + {corr_factor:.6f}')
print(f'                  = {X3*cross_factor:.6f} + {corr_factor:.6f}')
print(f'                  = {X3*cross_factor + corr_factor:.6f}')
print(f'  Check: {x_tau_mu_R3:.6f}  ✓')
print()

# The quark s/d uses outermost level directly — no projection needed
# So x_sd IS a genuine single-level exponent
print(f'STRUCTURAL CONCLUSION:')
print(f'  - x_eff(mu/e) = p2 = 3   : genuine single-level exponent (R3 lepton)')
print(f'  - x_eff(s/d)  = {x_sd_R3:.4f} : genuine single-level exponent (R3 quark)')
print(f'  - x_eff(tau/mu) = {x_tau_mu_R3:.4f} : PROJECTED from 2-level mechanism')
print(f'')
print(f'  The coincidence x_sd ≈ x_tau/mu is that a single-level quark process')
print(f'  produces the same effective exponent as a multi-level lepton process')
print(f'  projected onto the same basis. This is suggestive but may be accidental.')
print()

# What would make it NON-accidental? If x_sd = x_tau/mu exactly.
# Then: C0_q^x = C0_l^x from R3, dividing: x = ln(target ratio) / ln(C0 ratio)
# The quark/lepton C0 ratio on R3:
C0_ratio = C0_q_R3 / C0_l_R3
print(f'  QUARK/LEPTON C0 RATIO ON R3: {C0_ratio:.6f}')
print(f'  ln ratio: {np.log(C0_ratio):.6f}')
print(f'  The {diff_pct:+.4f}% difference means the exponents are NOT equal.')
print(f'  The coincidence is numerical, at the ~0.1% level.')


S3: THE EXPONENT COINCIDENCE

ALL EFFECTIVE EXPONENTS PROJECTED ONTO OUTERMOST C0:
  Lepton R3 C0 = 5.911955
  Quark  R3 C0 = 6.606742

  x_eff(mu/e)   = ln(206.77)/ln(5.911955) = 3.000376  [IDENTIFIED: p2 = 3]
  x_eff(tau/mu) = ln(16.817)/ln(5.911955) = 1.588310  [actual mechanism: R2 level]
  x_eff(s/d)    = ln(20.00)/ln(6.606742) = 1.586646  [OPEN]

THE NB134 COINCIDENCE (both projected onto outermost level):
  x_eff(s/d)    = 1.586646
  x_eff(tau/mu) = 1.588310
  Difference:     +0.1049%

DIAGNOSIS: WHY THE COINCIDENCE EXISTS
  The tau/mu actual mechanism (NB124): C0_R2^x3 * p3/p4
    Algebraic exponent x3 = 1.909859 (on R2, C0 = 5.227295)
    Correction: p3/p4 = 0.714286

  When PROJECTED onto R3:
    x_eff(tau/mu) = x3 * ln(C0_R2)/ln(C0_R3) + ln(p3/p4)/ln(C0_R3)
                  = 1.909859 * 0.930735 + -0.189351
                  = 1.777573 + -0.189351
                  = 1.588222
  Check: 1.588310  ✓

STRUCTURAL CONCLUSION:
  - x_eff(mu/e) = p2 = 3   : genuine single-level expo

## S4: Complete 9-Mass Table from $M_Z$

The complete chain from $M_Z$ to all 9 charged fermion masses:

**Up-type quarks**: $m_t$ (algebraic) → $m_c$ (cascade $m_t/m_c$) → $m_u$ (cascade $m_c/m_u$)

**Down-type quarks**: $m_b = m_t / 42$ → $m_s$ (cascade $m_b/m_s$) → $m_d$ (cascade $m_s/m_d$)

**Charged leptons**: $m_\mu/m_e$ (window-0, $C_0^3$) → $m_\tau/m_\mu$ (window-0 + $p_3/p_4$)

For the lepton masses, we need one absolute mass anchor. Use $m_e = 0.51100$ MeV (PDG).

In [9]:

# -- S4: Complete 9-mass table from M_Z --

# PDG 2024 reference masses (central values, uncertainties)
PDG = {
    't':   (172.69, 0.30),       # GeV
    'b':   (4.183, 0.007),       # GeV (MS-bar)
    'c':   (1.270, 0.020),       # GeV (MS-bar)
    's':   (93.4e-3, 8.6e-3),    # GeV (MS-bar)
    'd':   (4.67e-3, 0.48e-3),   # GeV (MS-bar)
    'u':   (2.16e-3, 0.49e-3),   # GeV (MS-bar)
    'tau': (1.77686, 0.00012),   # GeV
    'mu':  (0.105658, 0.0),      # GeV
    'e':   (0.000511, 0.0),      # GeV
}

print('S4: COMPLETE 9-MASS TABLE FROM M_Z')
print('=' * 75)
print()

# ARCHITECTURE NOTE:
# The quark and lepton sectors use DIFFERENT mass extraction methodologies.
# This is a structural feature of the four mass channels, not a deficiency.
#
# LEPTON sector: Window-0 CP ratios (T-independent, algebraic exponents)
#   - m_mu/m_e = C0_l_R3^p2                      [#277 PASS, NB135]
#   - m_tau/m_mu = C0_l_R2^x3 * p3/p4            [#269 PASS, NB124]
#
# QUARK sector: Cumulative pipeline (NB72, validated by NB81)
#   - m_s/m_d = R4^x4                             [#133 PASS, NB72]
#   - m_b/m_s = R2^x2                             [#135 PASS, NB72]
#   - m_c/m_u = R3^x3 * R4^x4                    [#134 PASS, NB72]
#   - m_t/m_c = R2^x2 * R3^x3 / R4^lam7         [#140 PASS, NB72]
#
# The NB72 cumulative pipeline uses a 50-branch subsample with specific
# sector accumulation. NB81 validated these values at T=5000.

# NB72/NB81 established quark mass ratios (zero free parameters)
NB72_QUARK = {
    'm_s/m_d': (19.92, '#133'),    # NB72: R4^x4
    'm_b/m_s': (45.83, '#135'),    # NB72: R2^x2
    'm_c/m_u': (627.4, '#134'),    # NB72: R3^x3 * R4^x4
    'm_t/m_c': (137.7, '#139-140'), # NB72: R2^x2 * R3^x3 / R4^lam7
}

# Window-0 lepton mass ratios (this notebook, confirmed T-independent)
W0_LEPTON = {
    'm_mu/m_e': (pred_mu_e, '#277'),       # C0^p2
    'm_tau/m_mu': (pred_tau_mu, '#269'),   # C0^x3 * p3/p4
}

# Algebraic anchors
ALG = {
    'm_t/M_Z': (p2**2 / np.sqrt(np.pi * p4), '#258'),   # NB118
    'm_t/m_b': (P4 // p3, '#271'),                        # NB127
}

# Print summary of mass channels
print('MASS CHANNEL INVENTORY:')
print()
print('  ALGEBRAIC:')
for name, (val, ident) in ALG.items():
    print(f'    {name:>12} = {val:>10.4f}   ({ident})')
print()
print('  QUARK (NB72 cumulative pipeline):')
for name, (val, ident) in NB72_QUARK.items():
    sm = target_value(name)
    dev = (val / sm - 1) * 100
    print(f'    {name:>12} = {val:>10.4f}   (SM: {sm:>10.3f}, {dev:>+7.3f}%,  {ident})')
print()
print('  LEPTON (window-0 channel formulas):')
for name, (val, ident) in W0_LEPTON.items():
    sm = target_value(name)
    dev = (val / sm - 1) * 100
    print(f'    {name:>12} = {val:>10.4f}   (SM: {sm:>10.3f}, {dev:>+7.3f}%,  {ident})')
print()

# --- Build absolute masses ---
# Top quark (algebraic anchor)
m_t = M_Z * ALG['m_t/M_Z'][0]
# Bottom quark (cross-sector)
m_b = m_t / ALG['m_t/m_b'][0]
# Charm (cascade from top)
m_c = m_t / NB72_QUARK['m_t/m_c'][0]
# Up (cascade from charm)
m_u = m_c / NB72_QUARK['m_c/m_u'][0]
# Strange (cascade from bottom)
m_s = m_b / NB72_QUARK['m_b/m_s'][0]
# Down (cascade from strange)
m_d = m_s / NB72_QUARK['m_s/m_d'][0]

# Leptons: window-0 channel formulas
m_e = PDG['e'][0]
m_mu = m_e * W0_LEPTON['m_mu/m_e'][0]
m_tau = m_mu * W0_LEPTON['m_tau/m_mu'][0]

# --- Full mass table ---
print('COMPLETE 9-MASS TABLE')
print(f'  Anchors: M_Z = {M_Z} GeV, m_e = {m_e*1e3:.3f} MeV')
print(f'  Free parameters: 0')
print()
print(f'  {"Fermion":>8} {"Solenoid":>14} {"PDG":>14} {"Dev %":>8} {"Dev σ":>8} {"Source":>18}')
print(f'  {"-"*78}')

solenoid_masses = [
    ('t',   m_t,   '#258 algebraic'),
    ('b',   m_b,   '#271 cross-sector'),
    ('c',   m_c,   'NB72 cascade'),
    ('s',   m_s,   'NB72 cascade'),
    ('d',   m_d,   'NB72 cascade'),
    ('u',   m_u,   'NB72 cascade'),
    ('tau', m_tau,  '#269 window-0'),
    ('mu',  m_mu,   '#277 window-0'),
    ('e',   m_e,    'anchor'),
]

devs = []
for name, sol, src in solenoid_masses:
    pdg_val, pdg_err = PDG[name]
    dev_pct = (sol / pdg_val - 1) * 100
    dev_sig = (sol - pdg_val) / pdg_err if pdg_err > 0 else 0.0
    if name != 'e':
        devs.append(abs(dev_pct))
    
    # Format mass string
    if sol > 1:
        s_str = f'{sol:.3f} GeV'
        p_str = f'{pdg_val:.3f} GeV'
    elif sol > 0.01:
        s_str = f'{sol*1e3:.3f} MeV'
        p_str = f'{pdg_val*1e3:.3f} MeV'
    else:
        s_str = f'{sol*1e3:.4f} MeV'
        p_str = f'{pdg_val*1e3:.4f} MeV'
    
    sig_str = f'{dev_sig:+.1f}σ' if pdg_err > 0 else '—'
    print(f'  {name:>8} {s_str:>14} {p_str:>14} {dev_pct:>+7.2f}% {sig_str:>8} {src:>18}')

devs = np.array(devs)
print()
print(f'  Mean |deviation| (8 fermions): {devs.mean():.2f}%')
print(f'  RMS  |deviation|:              {np.sqrt((devs**2).mean()):.2f}%')
print(f'  Max  |deviation|:              {devs.max():.2f}%')
print()

# Consistency check: m_s/m_d from cascade vs from m_b, m_s separately
m_s_d_derived = m_s / m_d
print(f'CONSISTENCY CHECKS:')
print(f'  m_s/m_d (from m_b chain): {m_s_d_derived:.4f} (NB72: {NB72_QUARK["m_s/m_d"][0]:.2f})')
# NB127 cross-check: (m_t/m_c)/(m_b/m_s) ≈ p2 = 3
cross_check = NB72_QUARK['m_t/m_c'][0] / NB72_QUARK['m_b/m_s'][0]
print(f'  (m_t/m_c)/(m_b/m_s) = {NB72_QUARK["m_t/m_c"][0]:.1f}/{NB72_QUARK["m_b/m_s"][0]:.2f} = {cross_check:.4f} ≈ p2 = {p2} (#272 PASS)')
print()

# Summary
print('METHODOLOGICAL SUMMARY:')
print('  The four mass channels operate in two distinct regimes:')
print()
print('  REGIME A — Window-0 (T-independent, algebraic):')
print(f'    Leptons:  m_mu/m_e ({pred_mu_e:.3f}, {(pred_mu_e/M_MU_E-1)*100:+.04f}%)')
print(f'              m_tau/m_mu ({pred_tau_mu:.4f}, {(pred_tau_mu/M_TAU_MU-1)*100:+.04f}%)')
print(f'    Precision: ~0.07% mean')
print()
print('  REGIME B — Cumulative pipeline (NB72, T-dependent):')
print(f'    Quarks: via multi-level exponents X4, X3, X2, LAM7')
print(f'    Precision: ~2% mean')
print()
print('  REGIME C — Algebraic (exact):')
print(f'    m_t/M_Z = p2^2/sqrt(pi*p4) ({m_t:.2f} GeV, +1.34%)')
print(f'    m_t/m_b = P4/p3 = 42 (-0.39%)')


S4: COMPLETE 9-MASS TABLE FROM M_Z

MASS CHANNEL INVENTORY:

  ALGEBRAIC:
         m_t/M_Z =     1.9192   (#258)
         m_t/m_b =    42.0000   (#271)

  QUARK (NB72 cumulative pipeline):
         m_s/m_d =    19.9200   (SM:     20.000,  -0.400%,  #133)
         m_b/m_s =    45.8300   (SM:     44.750,  +2.413%,  #135)
         m_c/m_u =   627.4000   (SM:    588.000,  +6.701%,  #134)
         m_t/m_c =   137.7000   (SM:    135.800,  +1.399%,  #139-140)

  LEPTON (window-0 channel formulas):
        m_mu/m_e =   206.6299   (SM:    206.768,  -0.067%,  #277)
      m_tau/m_mu =    16.8143   (SM:     16.817,  -0.016%,  #269)

COMPLETE 9-MASS TABLE
  Anchors: M_Z = 91.1876 GeV, m_e = 0.511 MeV
  Free parameters: 0

   Fermion       Solenoid            PDG    Dev %    Dev σ             Source
  ------------------------------------------------------------------------------
         t    175.007 GeV    172.690 GeV   +1.34%    +7.7σ     #258 algebraic
         b      4.167 GeV      4.183 GeV   -

## S5: Synthesis and Verdict

### The Four Mass Channels — Architecture Summary

The cascade produces fermion mass ratios through four distinct channels:

| Channel | Mechanism | Level | Exponent | Correction | Status |
|---------|-----------|-------|----------|------------|--------|
| Lepton intra-gen | Window-0 C₀^x | R₃ (outermost) | p₂ = 3 | — | PASS (#277) |
| Lepton inter-gen | Window-0 C₀^x × corr | R₂ (2nd-outer) | x₃ = λ(P₄)/(2π) | p₃/p₄ | PASS (#269) |
| Quark intra-gen | Cumulative pipeline | multi-level | x₄ = φ(P₄)/(2π) | — | PASS (NB72) |
| Cross-sector (t/b) | Algebraic | — | — | — | PASS (#271) |

**Note**: The quark intra-gen channel was ORIGINALLY established via the cumulative
pipeline (NB72) with exponents X₄, X₃, X₂. The window-0 effective exponent
x_eff ≈ 1.587 measures the same physics from a different angle but remains
algebraically unidentified.

### Open Frontier

The quark window-0 exponent (x ≈ 1.587) is T-independent and real but has
no clean algebraic identity. This is likely because the quark mass extraction
involves the CUMULATIVE pipeline (NB72), not a simple window-0 power law.
The effective exponent is a projection of a multi-level process onto a
single-level measurement — it need not be algebraically clean.

In [10]:

# -- S5: Compact mass table check + Scorecard --

# Quick deviation summary
print('S5: MASS TABLE VERIFICATION')
print('=' * 65)
for name, sol, src in solenoid_masses:
    pdg_val, _ = PDG[name]
    dev = (sol / pdg_val - 1) * 100
    if sol > 1:
        s = f'{sol:.3f} GeV'
    elif sol > 0.01:
        s = f'{sol*1e3:.3f} MeV'
    else:
        s = f'{sol*1e3:.4f} MeV'
    print(f'  {name:>4}: {s:>14}  ({dev:>+7.2f}%)  {src}')

print(f'\n  Mean |dev|: {devs.mean():.2f}%,  RMS: {np.sqrt((devs**2).mean()):.2f}%')
print()

# SCORECARD
print('NB136 SCORECARD')
print('=' * 65)
print()
print('FOUR MASS CHANNEL ARCHITECTURE — COLLECTED AND SYSTEMATIZED:')
print()
print('  Channel 1: Lepton intra-gen (m_mu/m_e)')
print(f'    Formula: C0_l_R3^p2 = {pred_mu_e:.3f}  (SM {M_MU_E:.3f}, -0.07%)')
print(f'    Status: #277 PASS. Exponent = p2 = 3.')
print()
print('  Channel 2: Lepton inter-gen (m_tau/m_mu)')
print(f'    Formula: C0_l_R2^x3 * p3/p4 = {pred_tau_mu:.4f}  (SM {M_TAU_MU:.3f}, -0.02%)')
print(f'    Status: #269 PASS. Uses R2 level + dissipation correction.')
print()
print('  Channel 3: Quark sector (cumulative, NB72)')
print(f'    Exponents: x4=phi(P4)/(2pi), x3=lam(P4)/(2pi), x2=phi(P3)/(2pi)')
print(f'    Multi-level architecture: R4 (gen1-2), R3 (inter-sector), R2 (gen2-3)')
print(f'    + cascade correction R4^(-lam7) for top quark')
print(f'    Status: #133-136, #139-140 PASS.')
print()
print('  Channel 4: Algebraic cross-sector')
print(f'    m_t/M_Z = p2^2/sqrt(pi*p4) = {m_t:.2f} GeV  (#258 PASS)')
print(f'    m_t/m_b = P4/p3 = 42                          (#271 PASS)')
print()
print('QUARK WINDOW-0 EXPONENT:')
print(f'  x_eff(s/d) = {x_eff_q:.6f} ≈ 2^(2/3) = {2**(2/3):.6f} (+0.048%)')
print(f'  4x less precise than lepton (475 ppm vs 125 ppm)')
print(f'  NOT promoted — quark mass extraction uses cumulative pipeline,')
print(f'  not single-level window-0 power law.')
print()
print('EXPONENT COINCIDENCE:')
print(f'  x_eff(s/d, R3) = {x_eff_q:.4f} ≈ x_eff(tau/mu, R3) = {x_tau_mu_R3:.4f}')
print(f'  Difference: {(x_tau_mu_R3/x_eff_q-1)*100:+.3f}%')
print(f'  NUMERICAL, not structural: different levels, channels, C0 values.')
print()
print('COMPLETE MASS TABLE: 8 masses + m_e anchor from M_Z')
print(f'  Mean |deviation|: {devs.mean():.2f}%')
print(f'  Zero free parameters')
print()
print('NEW IDENTITIES: 0')
print('  This is a synthesis notebook collecting and systematizing')
print('  established results across NB72, NB118, NB124, NB127, NB134-135.')
print()
print(f'Running total: 277 identities, 0 free parameters')


S5: MASS TABLE VERIFICATION
     t:    175.007 GeV  (  +1.34%)  #258 algebraic
     b:      4.167 GeV  (  -0.39%)  #271 cross-sector
     c:      1.271 GeV  (  +0.07%)  NB72 cascade
     s:     90.919 MeV  (  -2.66%)  NB72 cascade
     d:     4.5642 MeV  (  -2.27%)  NB72 cascade
     u:     2.0257 MeV  (  -6.22%)  NB72 cascade
   tau:      1.775 GeV  (  -0.08%)  #269 window-0
    mu:    105.588 MeV  (  -0.07%)  #277 window-0
     e:     0.5110 MeV  (  +0.00%)  anchor

  Mean |dev|: 1.64%,  RMS: 2.57%

NB136 SCORECARD

FOUR MASS CHANNEL ARCHITECTURE — COLLECTED AND SYSTEMATIZED:

  Channel 1: Lepton intra-gen (m_mu/m_e)
    Formula: C0_l_R3^p2 = 206.630  (SM 206.768, -0.07%)
    Status: #277 PASS. Exponent = p2 = 3.

  Channel 2: Lepton inter-gen (m_tau/m_mu)
    Formula: C0_l_R2^x3 * p3/p4 = 16.8143  (SM 16.817, -0.02%)
    Status: #269 PASS. Uses R2 level + dissipation correction.

  Channel 3: Quark sector (cumulative, NB72)
    Exponents: x4=phi(P4)/(2pi), x3=lam(P4)/(2pi), x2=phi(P

In [11]:

# Compact verification
print(f't={m_t:.2f} b={m_b:.3f} c={m_c:.3f} s={m_s*1e3:.1f}MeV d={m_d*1e3:.2f}MeV u={m_u*1e3:.2f}MeV')
print(f'tau={m_tau:.5f} mu={m_mu:.6f}')
print(f'Mean|dev|={devs.mean():.2f}%')


t=175.01 b=4.167 c=1.271 s=90.9MeV d=4.56MeV u=2.03MeV
tau=1.77539 mu=0.105588
Mean|dev|=1.64%
